# Cortex Unified ML Gateway (Ngrok Static Domain - AMD HACKATHON EDITION)

This notebook runs all major ML workloads (LLM, Embeddings, Document Parsing) and exposes them through a **single static Ngrok domain**.
It safely uses a Virtual Environment (`.venv`) that inherits your Hackathon's perfectly configured AMD ROCm + vLLM system packages!

In [ ]:
import os
import subprocess
import sys

print("Setting up Virtual Environment (inheriting your pre-installed vLLM + ROCm)...")
if not os.path.exists(".venv"):
    # CRITICAL: We use --system-site-packages so it keeps the Hackathon's vLLM 0.16.0 and PyTorch 2.9!
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", ".venv"], check=True)

pip_path = ".venv/bin/pip"
python_path = ".venv/bin/python"

print("Installing MISSING dependencies into .venv (keeping vLLM untouched)...")
subprocess.run([pip_path, "install", "-q", "pyngrok", "docling", "fastembed", "fastapi", "uvicorn", "python-multipart", "httpx"], check=True)
print("Dependencies installed successfully in .venv!")

In [ ]:
%%writefile cortex_ml_gateway.py
import uvicorn
import httpx
import os
import sys
import unicodedata
import re
import hashlib
from contextlib import asynccontextmanager
from fastapi import FastAPI, Request, UploadFile, File
from fastapi.responses import StreamingResponse
from pyngrok import ngrok
from fastembed import TextEmbedding
from docling.document_converter import DocumentConverter
from docling.chunking import HierarchicalChunker

# Force unbuffered output for prints in this script
sys.stdout.reconfigure(line_buffering=True)

# Lazy-loaded models
embedding_model = None
doc_converter = None
chunker = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    global embedding_model, doc_converter, chunker
    print("Loading FastEmbed...")
    embedding_model = TextEmbedding(model_name="BAAI/bge-base-en-v1.5", threads=4)
    print("Loading Docling...")
    doc_converter = DocumentConverter()
    print("Loading Hierarchical Chunker...")
    chunker = HierarchicalChunker()
    print("Unified Gateway Ready!")

    # Launch Ngrok Tunnel
    ngrok_token = os.getenv("NGROK_AUTH_TOKEN")
    ngrok_domain = os.getenv("NGROK_STATIC_DOMAIN")
    if ngrok_token and ngrok_domain:
        ngrok.set_auth_token(ngrok_token)
        # Clean the domain of https:// and trailing slashes to prevent PyngrokNgrokHTTPError 9038
        clean_domain = ngrok_domain.replace("https://", "").replace("http://", "").strip("/")
        public_url = ngrok.connect(8000, domain=clean_domain).public_url
        print("\n" + "="*60)
        print("\033[92mCORTEX ML GATEWAY IS LIVE!\033[0m")
        print("="*60)
        print(f"Add these variables to your Render .env file:")
        print(f"LLM_BASE_URL={public_url}/v1")
        print(f"FAST_MODEL_BASE_URL={public_url}/v1")
        print(f"EMBEDDING_MODEL_ENDPOINT={public_url}/v1")
        print(f"REMOTE_PARSER_URL={public_url}/parse")
        print("="*60 + "\n")
    else:
        print("WARNING: Ngrok Token or Domain not provided. Running locally only.")
    yield

app = FastAPI(title="Cortex Unified ML Gateway", lifespan=lifespan)

# --- ROUTE 1: Docling Parsing & Chunking ---
@app.post("/parse")
async def parse_document(file: UploadFile = File(...)):
    print(f"[Docling] Parsing document: {file.filename}")
    file_bytes = await file.read()
    temp_path = f"/tmp/{file.filename}"
    with open(temp_path, "wb") as f:
        f.write(file_bytes)
        
    result = doc_converter.convert(temp_path)
    
    print(f"[Docling] Chunking document: {file.filename}")
    chunks_iter = chunker.chunk(result.document)
    
    artifact_chunks = []
    for index, chunk in enumerate(chunks_iter):
        chunk_text = chunk.text
        chunk_meta = chunk.meta.export_json_dict() if chunk.meta else {}
        
        headings = chunk_meta.get("headings", [])
        heading_path = "/".join(headings) if headings else "root"
        
        page_numbers = []
        for item in chunk_meta.get("doc_items", []):
            for prov in item.get("prov", []):
                if "page_no" in prov:
                    page_numbers.append(str(prov["page_no"]))
        page_str = ",".join(sorted(set(page_numbers))) if page_numbers else "0"
        
        normalized_text = unicodedata.normalize("NFKC", chunk_text.strip())
        normalized_text = re.sub(r'\s+', ' ', normalized_text)
        
        artifact_chunks.append({
            "text": chunk_text,
            "headings": headings,
            "page_numbers": [int(p) for p in page_numbers] if page_numbers else [],
            "bbox": None,
            "chunk_index": index,
            "token_count": len(chunk_text.split()),
            "normalized_text": normalized_text,
            "heading_path": heading_path,
            "page_str": page_str
        })
    
    print(f"[Docling] Successfully processed {len(artifact_chunks)} chunks for {file.filename}")
    
    return {
        "markdown": result.document.export_to_markdown(),
        "metadata": {
            "origin": file.filename,
            "page_count": len(result.document.pages)
        },
        "page_count": len(result.document.pages),
        "chunks": artifact_chunks
    }

# --- ROUTE 2: FastEmbed Embeddings (OpenAI Compatible) ---
@app.post("/v1/embeddings")
async def create_embeddings(request: Request):
    body = await request.json()
    texts = body.get("input", [])
    if isinstance(texts, str):
        texts = [texts]
        
    print(f"[FastEmbed] Start embedding {len(texts)} chunks")
    embeddings = list(embedding_model.embed(texts))
    print(f"[FastEmbed] Finished embedding {len(texts)} chunks")
    
    data = []
    for i, vec in enumerate(embeddings):
        data.append({
            "object": "embedding",
            "embedding": vec.tolist(),
            "index": i
        })
        
    return {
        "object": "list",
        "data": data,
        "model": "BAAI/bge-base-en-v1.5",
        "usage": {"prompt_tokens": 0, "total_tokens": 0}
    }

# --- ROUTE 3: Proxy LLM to Background vLLM ---
client = httpx.AsyncClient(base_url="http://localhost:8001")

@app.api_route("/v1/{path:path}", methods=["GET", "POST", "PUT", "DELETE", "OPTIONS", "HEAD", "PATCH"])
async def proxy_vllm(path: str, request: Request):
    print(f"[vLLM Proxy] Routing /{path}")
    req = client.build_request(
        method=request.method,
        url=f"/v1/{path}",
        headers=request.headers.raw,
        content=await request.body()
    )
    response = await client.send(req, stream=True)
    return StreamingResponse(
        response.aiter_raw(),
        status_code=response.status_code,
        headers={k: v for k, v in response.headers.items() if k.lower() not in ('content-length', 'transfer-encoding')}
    )

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")


In [ ]:
import subprocess
import time
import sys
import os

# =========================================================================
# CONFIGURATION
# =========================================================================
# Get these from https://dashboard.ngrok.com/
os.environ["NGROK_AUTH_TOKEN"] = "3GJuOTVdG3lNMelCPSyOjUY9LOY_59Q7Y29ZzeWpvAE7b1N8u"
os.environ["NGROK_STATIC_DOMAIN"] = "https://entree-antiquely-rotting.ngrok-free.dev" # Make sure to not include https://

venv_python_path = ".venv/bin/python"

print("Starting vLLM Server in the background...")
# Decreased GPU memory util to 0.5 to prevent OOM conflicts with other hackathon notebook kernels!
llm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-7B-Instruct",
    "--port", "8001",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.5",
    "--disable-custom-all-reduce",
    "--enforce-eager"
]
llm_process = subprocess.Popen(llm_cmd, stdout=open('vllm.log', 'w'), stderr=subprocess.STDOUT)

print("Waiting 15 seconds for vLLM to initialize...")
time.sleep(15)

print("Starting Unified FastAPI Gateway...")
# The gateway script imports docling/ngrok, so it MUST run in the .venv
gateway_process = subprocess.Popen([venv_python_path, "-u", "cortex_ml_gateway.py"])

try:
    gateway_process.wait()
except KeyboardInterrupt:
    print("\nShutting down servers...")
    llm_process.terminate()
    gateway_process.terminate()
    print("Shutdown complete.")